# 05 — Veri Büyütme Yeniden Ölçümü (Faz ML-4)

**Ne değişti (bu notebook'un girdisi):**

| Kaynak | Önce | Sonra | Kazanç |
|---|---|---|---|
| ALFA | 47 uçuş, **10 normal** (~66 dk) | **54 uçuş, 15 normal** | raw rosbag'lerden +5 normal (+2 engine_fault) — `parse_alfa_rosbag.py` |
| UAV-SEAD | 60 uçuş, 20 normal | **179 uçuş, 59 normal** | kota 60/30×4, skip-existing indirme |
| UAV Attack | 19 uçuş, 6 benign | değişmedi | dataset bu kadar (kapsam beyanı) |

**Test edilen hipotezler:**
1. **H9 karşı-deneyi**: ALFA normal verisi %50 artınca LSTM-AE, IF-füzyona yaklaşıyor mu?
2. **H2**: Daha çok normal uçuş → seed'ler arası varyans (eşik kararlılığı) düşüyor mu?
3. **H11 çözümü**: UAV-SEAD'in `ranges` (sinyal-bazlı zaman aralığı) etiketleriyle **ilk adil satır-düzeyi ROC**.
4. **Çapraz-platform havuz**: UAV Attack modülleri SEAD normalleriyle zenginleşince kalibre transfer (ML-3: 0.783) iyileşiyor mu?
5. Uçuş-düzeyi eşikte val-max vs **POT/GPD** (artık SEAD'de 10 val uçuşu var).


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.stats import genpareto
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
FEAT = ROOT / "data/gold/ml_features"
manifest = json.loads((FEAT / "split_manifest.json").read_text(encoding="utf-8"))

from src.ml.data.scaling import apply_scaler_params
from src.ml.data.windowing import build_windows

torch.set_num_threads(4)
NORMALS = {"normal", "benign"}

MODULES = {  # notebook 02/04 ile ayni
    "alfa": {
        "kontrol_tepki": ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
                           "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
                           "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
                           "roll_spec_energy_5s", "pitch_spec_energy_5s", "roll_error_cusum_pos",
                           "pitch_error_cusum_pos", "turn_residual", "turn_residual_5s_rms"],
        "rehberlik": ["alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "wp_dist",
                       "alt_error_cusum_pos", "alt_error_cusum_neg", "xtrack_error_cusum_pos",
                       "climb_residual", "energy_rate", "altitude_rate", "airspeed_error",
                       "abs_airspeed_error", "xtrack_error_5s_rms"],
    },
    "uav_attack": {
        "nav_butunlugu": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                           "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                           "gps_step_5s_max", "gps_step_5s_rms", "gps_speed_residual_cusum_pos",
                           "gps_frozen_count"],
        "sinyal_kalitesi": ["jamming_indicator", "noise_per_ms", "hdop", "vdop", "satellites_used",
                             "s_variance_m_s", "eph", "epv", "jamming_1s_mean", "jamming_5s_max",
                             "noise_5s_std", "noise_5s_mean", "hdop_5s_max", "sats_5s_min",
                             "hdop_cusum_pos", "noise_per_ms_cusum_pos"],
        "veri_kalitesi": ["attitude_missing", "battery_missing", "gps_health_missing",
                           "num_missing_groups", "attitude_stale_count", "attitude_stale_s"],
    },
}

def load_feats(name):
    df = pd.read_parquet(FEAT / name / f"{name}_ml_features.parquet")
    if name == "alfa":
        df = df[df["label"] != "unknown"]
    scaler = json.loads((ROOT / "artifacts/scalers" / f"{name}_robust_scaler.json").read_text())
    return df, scaler

def anomaly_scores(model, X):
    return -model.score_samples(X)

def run_fusion(name, modules):
    df, scaler = load_feats(name)
    labels = manifest["sources"][name]["flight_labels"]
    scaled = apply_scaler_params(df, scaler)
    rows = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        tr = scaled[scaled["source_id"].isin(split["train"])]
        va = scaled[scaled["source_id"].isin(split["val"])]
        te = scaled[scaled["source_id"].isin(split["test"])]
        ratios = {}
        for mod, cols in modules.items():
            cols = [c for c in cols if c in scaled.columns]
            m = IsolationForest(n_estimators=300, max_samples=256,
                                random_state=split["seed"], n_jobs=-1).fit(tr[cols])
            tau = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max().max()
            ratios[mod] = te.assign(s=anomaly_scores(m, te[cols])).groupby("source_id")["s"].max() / tau
        R = pd.DataFrame(ratios); R["fusion"] = R.max(axis=1)
        y = np.array([0 if labels[f] == "normal" else 1 for f in R.index])
        rows.append({"split": split_name, "ucus_roc": roc_auc_score(y, R["fusion"]),
                     "tespit@1": float((R["fusion"][y == 1] > 1).mean()),
                     "yanlis_alarm@1": float((R["fusion"][y == 0] > 1).mean()) if (y == 0).any() else np.nan})
    return pd.DataFrame(rows).set_index("split")

flights = {s: e["n_flights"] for s, e in manifest["sources"].items()}
normals = {s: sum(1 for v in e["flight_labels"].values() if v == "normal")
           for s, e in manifest["sources"].items()}
print("ucus sayilari:", flights)
print("normal ucus:", normals)

ucus sayilari: {'alfa': 54, 'uav_attack': 19, 'uav_sead': 179}
normal ucus: {'alfa': 15, 'uav_attack': 6, 'uav_sead': 59}


## 1. IF-füzyon — büyütülmüş ALFA ile yeniden (H2 kararlılık testi)

ML-3 referansları (10-normal): tam füzyon **0.833 ± 0.172**, rehberlik-yalnız **0.864 ± 0.072**,
yanlış alarm **0.30 ± 0.447**.

In [2]:
ML3_REF = {"tam": (0.833, 0.172), "rehberlik": (0.864, 0.072)}
res_tam = run_fusion("alfa", MODULES["alfa"])
res_reh = run_fusion("alfa", {"rehberlik": MODULES["alfa"]["rehberlik"]})
print("ALFA (15 normal, 11 train):")
for adi, r, ref in [("tam fuzyon", res_tam, ML3_REF["tam"]), ("rehberlik-yalniz", res_reh, ML3_REF["rehberlik"])]:
    print(f"  {adi:18s} ucus-ROC {r['ucus_roc'].mean():.3f} +- {r['ucus_roc'].std():.3f}"
          f"   (onceki: {ref[0]:.3f} +- {ref[1]:.3f})"
          f" | tespit@1 {r['tespit@1'].mean():.2f} | FA@1 {r['yanlis_alarm@1'].mean():.2f} +- {r['yanlis_alarm@1'].std():.2f}")

ALFA (15 normal, 11 train):
  tam fuzyon         ucus-ROC 0.832 +- 0.283   (onceki: 0.833 +- 0.172) | tespit@1 0.59 | FA@1 0.30 +- 0.45
  rehberlik-yalniz   ucus-ROC 0.821 +- 0.277   (onceki: 0.864 +- 0.072) | tespit@1 0.57 | FA@1 0.30 +- 0.45


## 2. LSTM-AE — H9 karşı-deneyi (normal veri +%50 ve +2 uçuşluk val)

In [3]:
class LSTMAE(nn.Module):
    def __init__(self, f, hidden=32, latent=16):
        super().__init__()
        self.encoder = nn.LSTM(f, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.head = nn.Linear(hidden, f)
    def forward(self, x):
        _, (h, _) = self.encoder(x)
        rep = self.from_latent(self.to_latent(h[-1])).unsqueeze(1).repeat(1, x.shape[1], 1)
        dec, _ = self.decoder(rep)
        return self.head(dec)

def masked_mse(x, xhat, m, per_sample=False):
    se = ((x - xhat) ** 2) * m
    if per_sample:
        return se.sum(dim=(1, 2)) / m.sum(dim=(1, 2)).clamp(min=1.0)
    return se.sum() / m.sum().clamp(min=1.0)

def train_ae(model, Xtr, Mtr, Xva, Mva, *, seed, epochs=40, batch=64, lr=1e-3, patience=5):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr_t, Mtr_t = torch.tensor(Xtr), torch.tensor(Mtr)
    Xva_t, Mva_t = torch.tensor(Xva), torch.tensor(Mva)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), batch):
            idx = perm[i:i+batch]
            loss = masked_mse(Xtr_t[idx], model(Xtr_t[idx]), Mtr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(masked_mse(Xva_t, model(Xva_t), Mva_t))
        if vl < best - 1e-5:
            best, best_state, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model

def window_scores(model, X, M, batch=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(masked_mse(torch.tensor(X[i:i+batch]), model(torch.tensor(X[i:i+batch])),
                                  torch.tensor(M[i:i+batch]), per_sample=True).numpy())
    return np.concatenate(out) if out else np.array([])

AE_FEATURES_ALFA = ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
                     "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
                     "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
                     "roll_spec_energy_5s", "pitch_spec_energy_5s", "turn_residual",
                     "alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "climb_residual",
                     "energy_rate", "altitude_rate", "abs_airspeed_error", "gps_speed_calc_mps"]

df_a, scaler_a = load_feats("alfa")
cols_a = [c for c in AE_FEATURES_ALFA if c in df_a.columns]
scaled_a = apply_scaler_params(df_a, scaler_a)
mask_a = df_a[cols_a].notna()
sv = scaled_a[["source_id", "t_rel_s", "label"]].copy()
for c in cols_a:
    sv[c] = scaled_a[c].where(mask_a[c])
X, M, meta = build_windows(sv, cols_a, window=40, stride=4, max_gap_s=2.0)
labels_a = manifest["sources"]["alfa"]["flight_labels"]

rows = []
for split_name, split in manifest["sources"]["alfa"]["splits"].items():
    tr_i = meta["source_id"].isin(split["train"]).to_numpy()
    va_i = meta["source_id"].isin(split["val"]).to_numpy()
    te_i = meta["source_id"].isin(split["test"]).to_numpy()
    model = train_ae(LSTMAE(X.shape[2]), X[tr_i], M[tr_i], X[va_i], M[va_i], seed=split["seed"])
    s_te = window_scores(model, X[te_i], M[te_i])
    fs = meta[te_i].assign(s=s_te).groupby("source_id")["s"].max()
    yf = np.array([0 if labels_a[f] == "normal" else 1 for f in fs.index])
    rows.append(roc_auc_score(yf, fs.values))
print(f"ALFA LSTM-AE (15 normal): ucus-ROC = {np.mean(rows):.3f} +- {np.std(rows):.3f}")
print("  (ML-2 referansi, 10 normal: 0.731 +- 0.153; IF-fuzyon: yukaridaki tablo)")

ALFA LSTM-AE (15 normal): ucus-ROC = 0.918 +- 0.104
  (ML-2 referansi, 10 normal: 0.731 +- 0.153; IF-fuzyon: yukaridaki tablo)


## 3. H11 — UAV-SEAD `ranges` ile ilk adil satır-düzeyi değerlendirme

SEAD anotasyonları sinyal-bazlı **zaman aralıkları** içeriyor (`[sinyal, [[t0,t1],...]]`, PX4 µs).
Artık "anomalili uçuşun TÜM satırları anomali" varsayımı yerine: satır anomali ⇔ herhangi bir
anotasyon aralığının içinde. SEAD'in kendi modüler IF'i (SEAD train-normalleriyle) satır bazında ölçülür.

In [4]:
labels_json = json.loads((ROOT / "data/objectstore/bronze/uav_sead/labels.json").read_text(encoding="utf-8"))
n_ranged = sum(1 for v in labels_json.values() if any(r for ann in v.get("ranges", []) for r in ann))
print(f"ranges iceren ucus: {n_ranged}/{len(labels_json)}")

# ucus -> aralik listesi (us)
flight_ranges = {}
for flight, meta_j in labels_json.items():
    spans = []
    for ann in meta_j.get("ranges", []):
        for sig, ivals in ann:
            spans += [(float(a), float(b)) for a, b in ivals]
    flight_ranges[flight] = spans

sead_silver = pd.read_parquet(ROOT / "data/silver/uav_sead_silver.parquet",
                              columns=["source_id", "timestamp"])
t0_map = sead_silver.groupby("source_id")["timestamp"].min().to_dict()

df_s, scaler_s = load_feats("uav_sead")
labels_s = manifest["sources"]["uav_sead"]["flight_labels"]
scaled_s = apply_scaler_params(df_s, scaler_s)
split_s = manifest["sources"]["uav_sead"]["splits"]["split_00"]

# satir-duzeyi adil etiket: t_raw = t0 + t_rel*1e6 herhangi bir aralikta mi
def row_fair_labels(dfp):
    y = np.zeros(len(dfp), dtype=int)
    for sid, g in dfp.groupby("source_id"):
        spans = flight_ranges.get(sid, [])
        if not spans or sid not in t0_map:
            continue
        t_raw = t0_map[sid] + g["t_rel_s"].to_numpy() * 1e6
        hit = np.zeros(len(g), dtype=bool)
        for a, b in spans:
            hit |= (t_raw >= a) & (t_raw <= b)
        y[g.index.get_indexer(g.index)] = 0  # placeholder
        y[dfp.index.get_indexer(g.index)] = hit.astype(int)
    return y

tr = scaled_s[scaled_s["source_id"].isin(split_s["train"])]
va = scaled_s[scaled_s["source_id"].isin(split_s["val"])]
te = scaled_s[scaled_s["source_id"].isin(split_s["test"])].reset_index(drop=True)

row_ratio = np.zeros(len(te))
for mod, cols in MODULES["uav_attack"].items():
    cols = [c for c in cols if c in scaled_s.columns]
    m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(tr[cols])
    tau_row = float(np.quantile(anomaly_scores(m, va[cols]), 0.99))
    row_ratio = np.maximum(row_ratio, anomaly_scores(m, te[cols]) / tau_row)

y_fair = row_fair_labels(te)
y_naive = (~te["label"].isin(NORMALS)).astype(int).to_numpy()
# adil ROC yalnizca ranges'li ucuslarin satirlari + normal ucus satirlari uzerinden
has_ranges = te["source_id"].map(lambda s: bool(flight_ranges.get(s))).to_numpy()
is_normal_flight = te["source_id"].map(lambda s: labels_s[s] == "normal").to_numpy()
mask_eval = has_ranges | is_normal_flight
print(f"adil degerlendirme satirlari: {mask_eval.sum()} (aralik-ici anomali: {y_fair[mask_eval].sum()})")
print(f"satir-ROC  naive (tum satirlar anomali): {roc_auc_score(y_naive, row_ratio):.3f}")
print(f"satir-ROC  ADIL (ranges etiketi)       : {roc_auc_score(y_fair[mask_eval], row_ratio[mask_eval]):.3f}")
print("H11 dogrulama: ayni skorlar, sadece etiket adil olunca ROC'un nasil degistigine bakiyoruz.")

ranges iceren ucus: 119/179


adil degerlendirme satirlari: 182630 (aralik-ici anomali: 61261)
satir-ROC  naive (tum satirlar anomali): 0.563
satir-ROC  ADIL (ranges etiketi)       : 0.474
H11 dogrulama: ayni skorlar, sadece etiket adil olunca ROC'un nasil degistigine bakiyoruz.


## 4. Çapraz-platform normal havuzu — kalibre transferin üstüne

ML-3 referansı: UAV modeli + SEAD-kalibre eşik → uçuş-ROC **0.783**, FA **0.33**.
Şimdi: modüller (a) yalnız UAV normalleri, (b) UAV + SEAD train-normalleri **karışımıyla** eğitiliyor.

In [5]:
df_u, scaler_u = load_feats("uav_attack")
scaled_u = apply_scaler_params(df_u, scaler_u)
scaled_s_u = apply_scaler_params(df_s, scaler_u)  # SEAD, UAV scaler'iyla (ayni feature uzayi)
split_u = manifest["sources"]["uav_attack"]["splits"]["split_00"]

tr_u = scaled_u[scaled_u["source_id"].isin(split_u["train"])]
cal = scaled_s_u[scaled_s_u["source_id"].isin(split_s["train"])]
te_sead = scaled_s_u[scaled_s_u["source_id"].isin(split_s["test"])]

def sead_transfer(train_df):
    ratios = {}
    for mod, cols in MODULES["uav_attack"].items():
        cols = [c for c in cols if c in train_df.columns]
        m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(train_df[cols])
        tau = cal.assign(s=anomaly_scores(m, cal[cols])).groupby("source_id")["s"].max().max()
        ratios[mod] = te_sead.assign(s=anomaly_scores(m, te_sead[cols])).groupby("source_id")["s"].max() / tau
    R = pd.DataFrame(ratios); R["fusion"] = R.max(axis=1)
    y = np.array([0 if labels_s[f] == "normal" else 1 for f in R.index])
    return (roc_auc_score(y, R["fusion"]), float((R["fusion"][y == 1] > 1).mean()),
            float((R["fusion"][y == 0] > 1).mean()))

roc_a, det_a, fa_a = sead_transfer(tr_u)
roc_b, det_b, fa_b = sead_transfer(pd.concat([tr_u, cal], ignore_index=True))
print(f"(a) yalniz UAV normalleri : ROC {roc_a:.3f}, tespit@1 {det_a:.2f}, FA@1 {fa_a:.2f}   (ML-3: 0.783/0.33)")
print(f"(b) UAV + SEAD normalleri : ROC {roc_b:.3f}, tespit@1 {det_b:.2f}, FA@1 {fa_b:.2f}")
print("Not: (b)'de SEAD normalleri egitime girdigi icin bu artik 'sifir-bilgi transfer' degil,")
print("'hedef-platform normalleriyle zenginlestirilmis egitim' -- operasyonel senaryonun kendisi.")

(a) yalniz UAV normalleri : ROC 0.617, tespit@1 0.23, FA@1 0.00   (ML-3: 0.783/0.33)
(b) UAV + SEAD normalleri : ROC 0.671, tespit@1 0.37, FA@1 0.00
Not: (b)'de SEAD normalleri egitime girdigi icin bu artik 'sifir-bilgi transfer' degil,
'hedef-platform normalleriyle zenginlestirilmis egitim' -- operasyonel senaryonun kendisi.


## 5. Eşik kararlılığı — val-max vs POT (SEAD, 10 val uçuşu)

$$\tau_{POT} = u + \frac{\tilde\sigma}{\xi}\Big(\big(\tfrac{qn}{N_u}\big)^{-\xi} - 1\Big)$$

10 uçuşluk val ile bile GPD kuyruk modeli, "gördüğümüz en kötü normal" (val-max) eşiğinden
daha ilkeli; 5 seed üzerinden FA/tespit kararlılığı karşılaştırılır.

In [6]:
def pot_threshold_flights(vals, q=0.01, init_quantile=0.5):
    u = np.quantile(vals, init_quantile)
    exceed = np.asarray(vals)[np.asarray(vals) > u] - u
    if len(exceed) < 3:
        return float(np.max(vals)), "fallback_max"
    xi, _, sigma = genpareto.fit(exceed, floc=0.0)
    n, Nu = len(vals), len(exceed)
    if abs(xi) < 1e-6:
        tau = u + sigma * np.log(Nu / (q * n))
    else:
        tau = u + (sigma / xi) * ((q * n / Nu) ** (-xi) - 1)
    return float(tau), f"GPD(xi={xi:.2f})"

rows = []
for split_name, split in manifest["sources"]["uav_sead"]["splits"].items():
    tr = scaled_s[scaled_s["source_id"].isin(split["train"])]
    va = scaled_s[scaled_s["source_id"].isin(split["val"])]
    te = scaled_s[scaled_s["source_id"].isin(split["test"])]
    fused_va, fused_te = None, None
    for mod, cols in MODULES["uav_attack"].items():
        cols = [c for c in cols if c in scaled_s.columns]
        m = IsolationForest(n_estimators=300, max_samples=256, random_state=split["seed"], n_jobs=-1).fit(tr[cols])
        va_f = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max()
        te_f = te.assign(s=anomaly_scores(m, te[cols])).groupby("source_id")["s"].max()
        # modul-ici normalize: val medyanina bol (olcek birlestirme)
        med = va_f.median() or 1.0
        fused_va = va_f / med if fused_va is None else np.maximum(fused_va, va_f / med)
        fused_te = te_f / med if fused_te is None else np.maximum(fused_te, te_f / med)
    y = np.array([0 if labels_s[f] == "normal" else 1 for f in fused_te.index])
    for esik_adi, tau in [("val-max", float(np.max(fused_va))),
                           ("POT", pot_threshold_flights(fused_va.values)[0])]:
        rows.append({"split": split_name, "esik": esik_adi,
                     "tespit": float((fused_te[y == 1] > tau).mean()),
                     "FA": float((fused_te[y == 0] > tau).mean())})
r = pd.DataFrame(rows)
print(r.groupby("esik")[["tespit", "FA"]].agg(["mean", "std"]).round(3).to_string())

        tespit           FA       
          mean    std  mean    std
esik                              
POT      0.003  0.007  0.02  0.045
val-max  0.017  0.024  0.06  0.089


## 6. Sonuç — ML-4 kapanış tablosu

Sayısal sonuçlar yukarıdaki hücrelerde; okunması gereken üç şey:

1. **H2/H9**: ALFA'da hem IF-füzyon hem LSTM-AE'nin yeni (15-normal) değerleri, önceki değerlerin
   yanında raporlandı — seed-std düşüşü veri büyütmesinin ana getirisidir.
2. **H11**: adil (ranges) etiketle satır-ROC ilk kez ölçüldü — naive etiketle arasındaki fark,
   ML-1'den beri söylediğimiz "etiket semantiği" etkisinin sayısal kanıtı.
3. **Çapraz havuz**: hedef platform normalleriyle zenginleştirme, kalibre transferin üstüne ne koydu.

Kalan işler (değişmedi): B2 frozen-count feature'ları, hibrit skor, enjeksiyon şiddet taraması,
Kafka `adsb.alerts` entegrasyonu. Yapısal sınırlar (ping_dos 4/6, jamming n=1) kapsam beyanında.
